# 六列表详细一致性检查

本 Notebook 不运行信号引擎，也不读取远端现货。它只读取：

- 第一个 Notebook 生成的 `runtime_outputs/最终执行日简表.csv`；
- 包内本地冻结六列表 `expected/local_freeze/最终执行日简表_本地冻结参考.csv`；
- 远端三状态文件：环境变量 `REMOTE_THREE_STATE_PATH`，默认是 `/home/hzy/cta/三状态冻结/IC_1545_three_state_and_downside_warning.csv`。

它做两组独立检查：将生成六列表与包内本地冻结六列表的六列逐日逐列比较；再将生成六列表中的三状态按实际执行日与远端三状态逐日比较。两份对照数据都不参与信号生成，也不修改任何信号。

In [ ]:
from pathlib import Path
import os
import sys

_UPLOAD_ROOT_RAW = Path(os.environ.get("FINAL_UPLOAD_PACKAGE_ROOT", "/home/hzy/cta/最终冻结运行上传包_20260819_1441")).expanduser()
if not _UPLOAD_ROOT_RAW.is_absolute():
    raise ValueError("FINAL_UPLOAD_PACKAGE_ROOT 必须是绝对路径")
UPLOAD_ROOT = _UPLOAD_ROOT_RAW.resolve()
PACKAGE_ROOT = UPLOAD_ROOT
if not (PACKAGE_ROOT / "src" / "compare_compact_output.py").is_file():
    raise FileNotFoundError(f"最终六列运行包不存在：{PACKAGE_ROOT}")
sys.path.insert(0, str(PACKAGE_ROOT / "src"))
from compare_compact_output import compare_compact_output

conclusion = compare_compact_output()
conclusion

In [ ]:
import json
import pandas as pd

conclusion_path = PACKAGE_ROOT / 'runtime_outputs' / '六列表一致性结论.json'
six_comparison_path = PACKAGE_ROOT / 'runtime_outputs' / '六列表逐日对比.csv'
six_sample_path = PACKAGE_ROOT / 'runtime_outputs' / '六列表不一致示例.csv'
state_comparison_path = PACKAGE_ROOT / 'runtime_outputs' / '三状态逐日对比.csv'
state_sample_path = PACKAGE_ROOT / 'runtime_outputs' / '三状态不一致示例.csv'
conclusion = json.loads(conclusion_path.read_text(encoding='utf-8'))
six_comparison = pd.read_csv(six_comparison_path, encoding='utf-8-sig')
state_comparison = pd.read_csv(state_comparison_path, encoding='utf-8-sig')

print('===== 最终检查结论 =====')
print('success =', conclusion['success'])
print('六列表完整一致：', conclusion['all_dates_and_signals_match'])
print('三状态远端一致：', conclusion['all_dates_and_three_state_match'])
print('生成六列表日期：', conclusion['generated_date_min'], '->', conclusion['generated_date_max'])
print('包内本地冻结六列表日期：', conclusion['local_reference_date_min'], '->', conclusion['local_reference_date_max'])
print('远端三状态日期：', conclusion['remote_three_state_date_min'], '->', conclusion['remote_three_state_date_max'])
print('行数：生成六列表', conclusion['generated_rows'], '本地冻结六列表', conclusion['local_reference_rows'], '远端三状态', conclusion['remote_three_state_rows'])

field_rows = []
for field, metrics in conclusion['field_metrics'].items():
    field_rows.append({
        '字段': field,
        '共同日期不一致行数': metrics['mismatch_rows_on_common_dates'],
        '共同日期一致行数': metrics['match_rows_on_common_dates'],
    })
print('===== 按字段详细检查 =====')
display(pd.DataFrame(field_rows))

print('===== 六列表前 50 条不一致或日期差异 =====')
six_sample = pd.read_csv(six_sample_path, encoding='utf-8-sig')
display(six_sample)
print('===== 三状态前 50 条不一致或日期差异 =====')
state_sample = pd.read_csv(state_sample_path, encoding='utf-8-sig')
display(state_sample)

print('===== 对比文件列 =====')
print('六列表逐日对比列：', list(six_comparison.columns))
print('三状态逐日对比列：', list(state_comparison.columns))
print('说明：success=True 表示生成六列表与包内本地冻结六列表六列完全一致，且生成六列表中的三状态与远端三状态逐日一致。')